# 第3章　分布の特性値

統計学実践ワークブック 第3章に対応。  
**方針**：「この数値は何を言いたいのか」のイメージを先に固める。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
import matplotlib
try:
    matplotlib.font_manager.findfont('IPAexGothic', fallback_to_default=False)
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'

np.random.seed(42)

---
## 1. 期待値・中央値・最頻値

**直感：分布の「代表点」には3種類ある**

「この集団の年収は平均500万円」と言うとき、その「平均」は本当に典型的な人を代表しているか？  
富裕層が数人いるだけで「平均」は大きく引っ張られる。

| 名前 | 意味 | 特徴 |
|---|---|---|
| **期待値（平均）** | 重心。全員の合計 ÷ 人数 | 外れ値に引っ張られやすい |
| **中央値** | ちょうど真ん中の人の値 | 外れ値に強い |
| **最頻値** | 一番多い値（ヒストグラムの山の頂上） | カテゴリデータにも使える |

**歪んだ分布では3つが離れる**：
- 右寄り（right-skewed）：高所得者が裾を右に引っ張る → 平均 > 中央値 > 最頻値
- 左寄り（left-skewed）：低得点者が裾を左に引っ張る → 平均 < 中央値 < 最頻値

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    ('右寄り（年収分布）',  stats.lognorm(s=0.8, scale=np.exp(6)).rvs(2000),  '#e63946'),
    ('正規分布（試験の点数）', stats.norm(loc=60, scale=10).rvs(2000),           '#457b9d'),
    ('左寄り（満足度スコア）', stats.beta(a=8, b=2).rvs(2000) * 100,             '#2a9d8f'),
]

for ax, (title, data, col) in zip(axes, configs):
    mean   = np.mean(data)
    median = np.median(data)
    # 最頻値はカーネル密度推定のピーク
    kde = stats.gaussian_kde(data)
    x_range = np.linspace(data.min(), data.max(), 500)
    mode_approx = x_range[np.argmax(kde(x_range))]

    ax.hist(data, bins=40, density=True, color=col, alpha=0.5, edgecolor='white')
    ax.plot(x_range, kde(x_range), color=col, lw=2)

    ax.axvline(mean,        color='#e63946', lw=2.5, linestyle='-',  label=f'平均    = {mean:.1f}')
    ax.axvline(median,      color='#333',    lw=2.5, linestyle='--', label=f'中央値 = {median:.1f}')
    ax.axvline(mode_approx, color='#f4a261', lw=2.5, linestyle=':',  label=f'最頻値 ≈ {mode_approx:.1f}')

    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylabel('密度', fontsize=9)

plt.suptitle('期待値・中央値・最頻値：正規分布では一致、歪むと3つが離れる', fontsize=12)
plt.tight_layout()
plt.show()

# 年収の例で直感を確認
print('=== 右寄り分布（年収）での代表値 ===')
income = configs[0][1]
print(f'平均   = {np.mean(income):,.0f}万円  ← 高所得者に引き上げられる')
print(f'中央値 = {np.median(income):,.0f}万円  ← 真ん中の人')
print(f'平均 > 中央値 → 「平均以上の年収」の人は実は半分以下しかいない')

---
## 2. 四分位範囲 (IQR) と 分位点関数

**直感：IQR = 「真ん中50%の人たちが収まる幅」**

標準偏差は外れ値に弱い（1人の大富豪が分散を爆増させる）。  
IQR（四分位範囲）は中央の50%だけを見るので、外れ値に強い。

- Q1（第1四分位）= 下位25%の境界
- Q3（第3四分位）= 下位75%の境界  
- **IQR = Q3 − Q1**（真ん中の50%が収まる幅）

**分位点関数（パーセンタイル関数）** は CDF の逆関数。  
「下位 p% の境界値は何か」→ `ppf(p)` で答えが出る。  
例：上位5%に入るには何点必要か？ → `ppf(0.95)` を計算。

In [ ]:
# 2つのデータ：外れ値なし vs 外れ値あり
normal_data  = stats.norm(60, 10).rvs(200)
outlier_data = np.append(normal_data, [200, 210, 220])  # 極端な外れ値を追加

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, label in [
    (axes[0], normal_data,  '外れ値なし'),
    (axes[1], outlier_data, '外れ値あり（200点台が3人）'),
]:
    q1, q2, q3 = np.percentile(data, [25, 50, 75])
    iqr = q3 - q1
    mean_v = np.mean(data)
    std_v  = np.std(data)

    ax.hist(data, bins=30, color='#457b9d', alpha=0.6, edgecolor='white', density=True)

    # IQR の帯
    ax.axvspan(q1, q3, alpha=0.2, color='#2a9d8f', label=f'IQR = {iqr:.1f}（Q1〜Q3）')
    ax.axvline(q2,    color='#2a9d8f', lw=2.5, linestyle='--', label=f'中央値 Q2 = {q2:.1f}')

    # ±1SD の帯
    ax.axvspan(mean_v - std_v, mean_v + std_v, alpha=0.15, color='#e63946',
               label=f'±1SD = [{mean_v-std_v:.1f}, {mean_v+std_v:.1f}]')
    ax.axvline(mean_v, color='#e63946', lw=2.5, linestyle='-', label=f'平均 = {mean_v:.1f}')

    ax.set_title(f'{label}\nIQR={iqr:.1f}、SD={std_v:.1f}', fontsize=11)
    ax.set_xlabel('点数', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('IQR vs 標準偏差：外れ値に強いのは IQR', fontsize=12)
plt.tight_layout()
plt.show()

print('外れ値なし → IQR={:.1f}, SD={:.1f}'.format(
    np.percentile(normal_data,75)-np.percentile(normal_data,25), np.std(normal_data)))
print('外れ値あり → IQR={:.1f}, SD={:.1f}'.format(
    np.percentile(outlier_data,75)-np.percentile(outlier_data,25), np.std(outlier_data)))
print('→ SD は外れ値で大きく変化、IQR はほぼ変わらない')

print('\n=== 分位点関数（ppf）: 「下位 p% の境界値は？」===')
dist_exam = stats.norm(60, 10)
for p in [0.25, 0.50, 0.75, 0.90, 0.95]:
    print(f'  下位 {p*100:.0f}% の境界 = {dist_exam.ppf(p):.1f}点')

---
## 3. 変動係数 (CV)

**直感：「平均と比べてどのくらいばらつくか」**

標準偏差 5cm は「身長のばらつき」として大きいか小さいか？  
→ 平均 170cm なら 5/170 ≈ 3%（小さい）  
→ もし平均が 5cm の虫なら 5/5 = 100%（巨大なばらつき）

$$\text{CV} = \frac{\text{標準偏差}}{\text{平均}}$$

**用途**：
- 単位が違う変数同士のばらつきを比較できる
- 例：「身長のばらつき」と「年収のばらつき」を同じ土俵で比べる

In [ ]:
# 異なる変数のばらつきを CV で比較
np.random.seed(0)
variables = {
    '身長 (cm)':     stats.norm(170, 7).rvs(500),
    '体重 (kg)':     stats.norm(65, 12).rvs(500),
    '年収 (万円)':   stats.lognorm(s=0.6, scale=np.exp(6)).rvs(500),
    '通勤時間 (分)': stats.gamma(a=2, scale=20).rvs(500),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左：各変数のヒストグラム（正規化）
colors = ['#457b9d', '#e63946', '#2a9d8f', '#f4a261']
ax1 = axes[0]
for (name, data), col in zip(variables.items(), colors):
    # z-score で正規化して重ねる
    z = (data - data.mean()) / data.std()
    ax1.hist(z, bins=30, density=True, alpha=0.4, color=col, label=name, edgecolor='white')
ax1.set_xlabel('Z スコア（平均0・SD1に正規化）', fontsize=10)
ax1.set_title('異なる変数を同じ軸で並べると\nばらつきの大小が分かりにくい', fontsize=10)
ax1.legend(fontsize=8)

# 右：CV の棒グラフで比較
ax2 = axes[1]
names, cvs, means, stds = [], [], [], []
for name, data in variables.items():
    cv = data.std() / data.mean()
    names.append(name)
    cvs.append(cv)
    means.append(data.mean())
    stds.append(data.std())

bars = ax2.bar(names, cvs, color=colors, alpha=0.8, edgecolor='white')
for bar, cv, mean, std in zip(bars, cvs, means, stds):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f'CV={cv:.2f}\n(SD={std:.1f}\n/Mean={mean:.0f})',
             ha='center', fontsize=7.5)

ax2.set_ylabel('変動係数 CV = SD / 平均', fontsize=10)
ax2.set_title('CV で比較：年収と通勤時間は\n平均に対して大きくばらつく', fontsize=10)
ax2.set_ylim(0, max(cvs) * 1.5)

plt.suptitle('変動係数：単位が違う変数のばらつきを同じ土俵で比較', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. 共分散・相関係数

**直感：「2つの変数が一緒に動くか」**

**共分散**：一緒に動く傾向の「方向と大きさ」。  
ただし単位に依存する（身長をmで測るかcmで測るかで値が変わる）。

**相関係数**：共分散を −1〜+1 に正規化したもの。単位に依存しない。

| 相関係数 r の値 | 意味 |
|---|---|
| +1.0 | 完全な正の相関（片方が上がれば必ず上がる） |
| +0.7 | 強い正の相関 |
| 0.0 | 無相関 |
| −0.7 | 強い負の相関 |
| −1.0 | 完全な負の相関 |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

rho_vals = [0.95, 0.5, 0.0, -0.5, -0.95, 0.0]
titles   = ['強い正の相関\nr≈+0.95', '中程度の正\nr≈+0.5',
            '無相関\nr≈0.0',         '中程度の負\nr≈-0.5',
            '強い負の相関\nr≈-0.95',  '非線形関係\n（相関係数は無意味）']

for ax, rho, title in zip(axes.flat, rho_vals, titles):
    if title.startswith('非線形'):
        # 非線形（U字形）：r≈0 だが明確に関係あり
        x = np.random.uniform(-3, 3, 300)
        y = x**2 + np.random.normal(0, 0.5, 300)
    else:
        cov_m = [[1, rho], [rho, 1]]
        xy = np.random.multivariate_normal([0, 0], cov_m, 300)
        x, y = xy[:, 0], xy[:, 1]

    r = np.corrcoef(x, y)[0, 1]
    col = '#e63946' if r > 0.3 else ('#457b9d' if r < -0.3 else '#888')

    ax.scatter(x, y, alpha=0.3, s=15, color=col)
    # 回帰直線
    if not title.startswith('非線形'):
        m, b = np.polyfit(x, y, 1)
        xr = np.linspace(x.min(), x.max(), 100)
        ax.plot(xr, m*xr+b, color=col, lw=2)
    ax.set_title(f'{title}\n実測 r={r:.2f}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('相関係数 r：−1〜+1 の数値で「一緒に動くか」を表す\n（注：r=0 でも非線形な関係はある）', fontsize=12)
plt.tight_layout()
plt.show()

# 共分散と相関係数の違い：単位変換の影響
print('=== 共分散は単位に依存するが、相関係数は依存しない ===')
h_cm = stats.norm(170, 7).rvs(500)
w_kg = 0.5 * h_cm - 22 + stats.norm(0, 8).rvs(500)

print(f'身長(cm)×体重(kg) の共分散 = {np.cov(h_cm, w_kg)[0,1]:.2f}')
h_m = h_cm / 100  # mに変換
print(f'身長( m)×体重(kg) の共分散 = {np.cov(h_m,  w_kg)[0,1]:.4f}  ← 100分の1に変化')
print(f'相関係数 (cm版) = {np.corrcoef(h_cm, w_kg)[0,1]:.4f}')
print(f'相関係数 ( m版) = {np.corrcoef(h_m,  w_kg)[0,1]:.4f}  ← 変わらない')

---
## 5. 疑似相関・偏相関係数

**疑似相関の直感：「2つが相関しているように見えるが、実は第3の変数が原因」**

有名な例：**アイスクリームの売上と水難事故数**は強く相関する。  
→ アイスが原因？ No。どちらも**夏（気温）**が原因。

エンジニア的に言えば：「共通の上流ノードがある」状態。

**偏相関係数**：第3の変数（交絡変数）の影響を除去した「本当の相関」。  
「気温を固定したとき、アイスと水難事故に相関は残るか？」→ ほぼゼロ。

In [ ]:
np.random.seed(7)
n = 200

# 気温（夏は高い）が共通原因
temperature = stats.norm(25, 8).rvs(n)  # 気温

# アイス売上 = 気温 + ノイズ
ice_cream = 2 * temperature + stats.norm(0, 5).rvs(n)

# 水難事故 = 気温 + ノイズ（アイスとは直接関係なし）
drowning = 1.5 * temperature + stats.norm(0, 6).rvs(n)

# 相関係数
r_ice_drown = np.corrcoef(ice_cream, drowning)[0, 1]
print(f'アイス売上 × 水難事故の相関係数 = {r_ice_drown:.3f}  ← 強い！（疑似相関）')

# 偏相関係数：気温の影響を除いてからの相関
# 残差 = (気温で説明できない部分)
def residuals(y, x):
    slope, intercept, _, _, _ = stats.linregress(x, y)
    return y - (slope * x + intercept)

res_ice   = residuals(ice_cream, temperature)
res_drown = residuals(drowning, temperature)
partial_r = np.corrcoef(res_ice, res_drown)[0, 1]
print(f'偏相関係数（気温を除いた後）       = {partial_r:.3f}  ← ほぼゼロ！')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 左：見かけの相関
ax1 = axes[0]
sc = ax1.scatter(ice_cream, drowning, c=temperature, cmap='RdYlBu_r',
                 s=40, alpha=0.7)
plt.colorbar(sc, ax=ax1, label='気温 (°C)')
m, b = np.polyfit(ice_cream, drowning, 1)
xr = np.linspace(ice_cream.min(), ice_cream.max(), 100)
ax1.plot(xr, m*xr+b, 'k--', lw=2)
ax1.set_xlabel('アイス売上', fontsize=10); ax1.set_ylabel('水難事故数', fontsize=10)
ax1.set_title(f'見かけの相関\nr = {r_ice_drown:.2f}（強い！）\nでも色を見ると…気温が原因', fontsize=10)

# 中：気温とアイス / 気温と水難
ax2 = axes[1]
ax2.scatter(temperature, ice_cream, alpha=0.4, s=20, color='#e63946',
            label=f'気温×アイス r={np.corrcoef(temperature,ice_cream)[0,1]:.2f}')
ax2.scatter(temperature, drowning, alpha=0.4, s=20, color='#457b9d',
            label=f'気温×水難 r={np.corrcoef(temperature,drowning)[0,1]:.2f}')
ax2.set_xlabel('気温 (°C)', fontsize=10)
ax2.set_title('どちらも気温と相関している\n→ 気温が「共通原因」', fontsize=10)
ax2.legend(fontsize=8)

# 右：残差の散布図（偏相関）
ax3 = axes[2]
ax3.scatter(res_ice, res_drown, alpha=0.4, s=25, color='#2a9d8f')
ax3.axhline(0, color='gray', lw=1, linestyle='--')
ax3.axvline(0, color='gray', lw=1, linestyle='--')
ax3.set_xlabel('アイス売上の残差\n（気温の影響を除いた部分）', fontsize=9)
ax3.set_ylabel('水難事故の残差', fontsize=9)
ax3.set_title(f'偏相関係数 = {partial_r:.3f}\n気温を除いたら無相関に', fontsize=10)

plt.suptitle('疑似相関と偏相関係数：「共通原因」を除いた本当の相関', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. 条件付き期待値・条件付き分散

**直感：「ある情報がわかったとき、期待値・ばらつきはどう変わるか」**

「全員の年収の平均」と「経験年数10年の人の年収の平均」は違う。  
条件付き期待値は「グループを絞り込んだときの平均」。

- **条件付き期待値** $E[Y|X=x]$：$X=x$ のグループの $Y$ の平均
- **条件付き分散** $\text{Var}[Y|X=x]$：$X=x$ のグループの $Y$ のばらつき

**全分散の法則**（重要）：
> 全体のばらつき = 「グループ内ばらつきの平均」 + 「グループ平均のばらつき」

In [ ]:
np.random.seed(1)
n = 1000
experience = np.random.randint(1, 21, n)  # 経験年数 1〜20年
# 年収 = 300 + 20×経験年数 + ノイズ（ベテランほど個人差が大きい）
noise_std = 20 + experience * 3  # 経験が長いほどばらつきも大きい
salary = 300 + 20 * experience + np.random.normal(0, noise_std)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：散布図 + 条件付き期待値の線
ax1 = axes[0]
ax1.scatter(experience, salary, alpha=0.2, s=15, color='#457b9d')

exp_bins = np.arange(1, 22)
cond_means = [salary[experience == e].mean() for e in exp_bins if np.sum(experience==e) > 0]
cond_stds  = [salary[experience == e].std()  for e in exp_bins if np.sum(experience==e) > 0]
valid_exp  = [e for e in exp_bins if np.sum(experience==e) > 0]

ax1.plot(valid_exp, cond_means, color='#e63946', lw=3,
         label='E[年収 | 経験年数=x]（条件付き期待値）')
ax1.fill_between(valid_exp,
                 np.array(cond_means) - np.array(cond_stds),
                 np.array(cond_means) + np.array(cond_stds),
                 alpha=0.2, color='#e63946', label='±1SD（条件付き分散）')

ax1.set_xlabel('経験年数', fontsize=11)
ax1.set_ylabel('年収（万円）', fontsize=11)
ax1.set_title('経験年数が増えるほど\n平均年収↑ かつ 個人差も↑', fontsize=11)
ax1.legend(fontsize=8)

# 右：全分散の法則を可視化
ax2 = axes[1]
total_var   = np.var(salary)
# グループ内分散の期待値
within_var  = np.mean([np.var(salary[experience == e])
                       for e in range(1, 21) if np.sum(experience==e) > 1])
# グループ平均の分散
between_var = np.var(cond_means)

bars = ax2.bar(
    ['全体の分散\nVar(Y)', 'グループ内分散\nE[Var(Y|X)]', 'グループ間分散\nVar(E[Y|X])'],
    [total_var, within_var, between_var],
    color=['#6a4c93', '#457b9d', '#e63946'], alpha=0.8, edgecolor='white'
)
for bar, v in zip(bars, [total_var, within_var, between_var]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
             f'{v:.0f}', ha='center', fontsize=10)

ax2.set_ylabel('分散', fontsize=11)
ax2.set_title(f'全分散の法則\n全体({total_var:.0f}) ≈ 内({within_var:.0f}) + 間({between_var:.0f})', fontsize=11)

plt.suptitle('条件付き期待値と分散：「グループを絞り込んだときの平均・ばらつき」', fontsize=12)
plt.tight_layout()
plt.show()

print(f'全分散の法則: {total_var:.1f} ≈ {within_var:.1f} + {between_var:.1f} = {within_var+between_var:.1f}')

---
## 7. 加重平均・幾何平均・調和平均

**直感：「どんな平均を使うべきかは状況による」**

「平均」は1種類ではない。問題の性質に合った平均を選ぶ必要がある。

| 平均の種類 | いつ使う | 例 |
|---|---|---|
| **加重平均** | 重要度が違うとき | テストの科目ごとに配点が違う場合 |
| **幾何平均** | 掛け算で積み上がるとき | 年率成長率の平均 |
| **調和平均** | 速度・レートを平均するとき | 往復の平均速度 |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- 加重平均：配点の違うテスト ---
ax1 = axes[0]
subjects = ['数学\n(配点3)', '英語\n(配点2)', '国語\n(配点1)']
scores   = [70, 90, 50]
weights  = [3, 2, 1]

simple_mean   = np.mean(scores)
weighted_mean = np.average(scores, weights=weights)

ax1.bar(subjects, scores, color=['#e63946', '#457b9d', '#2a9d8f'], alpha=0.7, edgecolor='white')
ax1.axhline(simple_mean,   color='gray',    lw=2, linestyle='--', label=f'単純平均 = {simple_mean:.1f}')
ax1.axhline(weighted_mean, color='#f4a261', lw=2.5, linestyle='-', label=f'加重平均 = {weighted_mean:.1f}')
ax1.set_ylabel('点数', fontsize=10)
ax1.set_title('加重平均：配点を考慮\n数学が得意なら加重平均↑', fontsize=10)
ax1.legend(fontsize=8)

# --- 幾何平均：投資リターン ---
ax2 = axes[1]
annual_returns = [1.20, 0.85, 1.30, 0.90, 1.25]  # 各年の成長率（1.2 = +20%）
years = range(1, len(annual_returns)+1)

arith_mean = np.mean(annual_returns)
geo_mean   = stats.gmean(annual_returns)

# 累積成長
cumulative_actual = np.cumprod(annual_returns)
cumulative_arith  = arith_mean ** np.arange(1, 6)
cumulative_geo    = geo_mean   ** np.arange(1, 6)

ax2.plot(years, cumulative_actual, 'o-', color='#457b9d', lw=2, label='実際の累積')
ax2.plot(years, cumulative_arith,  's--', color='#e63946', lw=1.5,
         label=f'算術平均({arith_mean:.3f})で推定')
ax2.plot(years, cumulative_geo,    '^--', color='#2a9d8f', lw=1.5,
         label=f'幾何平均({geo_mean:.3f})で推定')
ax2.set_xlabel('年数', fontsize=10)
ax2.set_ylabel('初期投資の倍率', fontsize=10)
ax2.set_title('幾何平均：投資の複利成長\n算術平均は実際より過大推定', fontsize=10)
ax2.legend(fontsize=7.5)

# --- 調和平均：往復速度 ---
ax3 = axes[2]
v_go   = 60  # 行きの速度 km/h
v_back = 40  # 帰りの速度 km/h
dist   = 120 # 片道の距離 km

# 本当の平均速度
total_time = dist/v_go + dist/v_back
real_avg   = 2*dist / total_time

arith_v = (v_go + v_back) / 2
harm_v  = stats.hmean([v_go, v_back])

ax3.bar(['行き', '帰り'], [v_go, v_back], color=['#e63946', '#457b9d'], alpha=0.7, edgecolor='white')
ax3.axhline(arith_v, color='gray',    lw=2, linestyle='--', label=f'算術平均 = {arith_v:.1f} km/h')
ax3.axhline(harm_v,  color='#f4a261', lw=2.5, linestyle='-', label=f'調和平均 = {harm_v:.1f} km/h')
ax3.axhline(real_avg, color='#2a9d8f', lw=1.5, linestyle=':', label=f'実際の平均 = {real_avg:.1f} km/h')
ax3.set_ylabel('速度 (km/h)', fontsize=10)
ax3.set_title(f'調和平均：往復の平均速度\n帰りは遅いので算術平均は過大推定', fontsize=10)
ax3.legend(fontsize=8)

plt.suptitle('3種類の平均：状況に合わせて使い分ける', fontsize=12)
plt.tight_layout()
plt.show()

print(f'算術平均 ≥ 幾何平均 ≥ 調和平均 （等号は全ての値が等しいとき）')
print(f'投資例: 算術={arith_mean:.4f}, 幾何={geo_mean:.4f}')
print(f'速度例: 算術={arith_v:.2f}, 調和={harm_v:.2f}, 実際={real_avg:.2f}')

---
## 8. 期待値ベクトル・分散共分散行列・相関係数行列

**直感：「複数変数の統計量を行列にまとめる」**

1変数なら「平均」「分散」で済む。  
複数変数になると変数の数だけ「平均」があり、変数のペアごとに「共分散」がある。

- **平均ベクトル**：各変数の期待値を縦に並べたベクトル
- **分散共分散行列**：対角が各変数の分散、非対角が変数ペアの共分散
- **相関係数行列**：分散共分散行列を正規化（対角が1、非対角が−1〜+1）

**エンジニア的に言えば**：1変数の「平均・分散」を多変数に拡張したデータ構造。

In [ ]:
np.random.seed(5)
n = 500

# 身長・体重・年齢・年収の4変数データ
age    = stats.norm(35, 8).rvs(n)
height = stats.norm(170, 7).rvs(n)
weight = 0.5*height - 22 + stats.norm(0, 8).rvs(n)
salary = 200 + 15*age + 0.5*weight + stats.norm(0, 80).rvs(n)

df = pd.DataFrame({
    '身長(cm)': height,
    '体重(kg)': weight,
    '年齢(歳)': age,
    '年収(万)': salary,
})

mean_vec = df.mean()
cov_mat  = df.cov()
corr_mat = df.corr()

print('=== 平均ベクトル（各変数の期待値）===')
print(mean_vec.round(2))

print('\n=== 分散共分散行列 ===')
print('（対角 = 各変数の分散、非対角 = 変数ペアの共分散）')
print(cov_mat.round(2))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：分散共分散行列のヒートマップ
ax1 = axes[0]
sns.heatmap(cov_mat, annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, ax=ax1, square=True,
            linewidths=1, cbar_kws={'shrink': 0.8})
ax1.set_title('分散共分散行列\n対角＝分散（単位に依存）\n非対角＝共分散', fontsize=10)

# 右：相関係数行列のヒートマップ
ax2 = axes[1]
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)  # 上三角を隠す
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax2, square=True,
            linewidths=1, cbar_kws={'shrink': 0.8})
ax2.set_title('相関係数行列\n対角＝1（自分との相関）\n非対角＝−1〜+1', fontsize=10)

plt.suptitle('期待値ベクトル・分散共分散行列・相関係数行列：多変量の統計量を一括管理', fontsize=11)
plt.tight_layout()
plt.show()

print('\n=== 相関係数行列 ===')
print('（単位を統一して比較しやすくしたもの）')
print(corr_mat.round(3))

---
## まとめ

| 概念 | 直感的な意味 |
|---|---|
| 期待値・中央値・最頻値 | 分布の「代表点」3種。歪んだ分布では大きく離れる |
| IQR | 真ん中50%の幅。外れ値に強い |
| 分位点関数 | CDF の逆。「下位p%の境界値は何か」を答える |
| 変動係数 | SD÷平均。単位が違う変数のばらつきを比較できる |
| 共分散 | 一緒に動く傾向（単位依存） |
| 相関係数 | 共分散を −1〜+1 に正規化（単位非依存） |
| 疑似相関 | 2変数の相関が「共通原因」のせいである状態 |
| 偏相関係数 | 第3変数の影響を除いた「本当の相関」 |
| 条件付き期待値 | グループを絞り込んだときの平均 |
| 条件付き分散 | グループを絞り込んだときのばらつき |
| 加重平均 | 重みが違うときの平均 |
| 幾何平均 | 掛け算で積み上がる量（成長率）の平均 |
| 調和平均 | 速度・レートの平均 |
| 平均ベクトル | 各変数の平均を縦に並べたもの |
| 分散共分散行列 | 全変数の分散＋ペアの共分散を行列にまとめたもの |
| 相関係数行列 | 分散共分散行列を正規化したもの（対角は1） |

**次章**：第4章「変数変換」